# Example 16.6 / 1: Solve and Test a 4 × 4 Nonogram

Companion notebook for **16.6 Nonograms** in the AMPL lab Documentation, section "Solve and Test a 4 × 4 Nonogram". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve and Test a 4 × 4 Nonogram

The model below implements the start-position formulation: `y` and `x` choose row and column block starts, `RowOrder` and `ColOrder` keep blocks separated, and `RowCover` and `ColCover` are the two exact coverage equalities. `DifferentImage` is the cell-pattern no-good, active only when `use_cut = 1`. The data sets clue counts and lengths with `ampl.param[...]`; the run solves with HiGHS, renders `z` as `#` and `.`, stores the image in `First`, and solves again with the cut:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param R integer >= 1;  # number of rows
param C integer >= 1;  # number of columns
set ROWS := 1..R;  # grid rows
set COLS := 1..C;  # grid columns
param row_blocks {ROWS} integer >= 0 default 0;  # number of blocks in each row clue
param col_blocks {COLS} integer >= 0 default 0;  # number of blocks in each column clue
param row_len {r in ROWS, 1..row_blocks[r]} integer >= 1;  # length of block k in row r
param col_len {c in COLS, 1..col_blocks[c]} integer >= 1;  # length of block k in column c

# start columns that leave room for the other blocks
set ROW_STARTS {r in ROWS, k in 1..row_blocks[r]} :=
    1 + sum {h in 1..k-1} (row_len[r,h] + 1) ..
    C - sum {h in k..row_blocks[r]} row_len[r,h] - (row_blocks[r] - k) + 1;
# start rows that leave room for the other blocks
set COL_STARTS {c in COLS, k in 1..col_blocks[c]} :=
    1 + sum {h in 1..k-1} (col_len[c,h] + 1) ..
    R - sum {h in k..col_blocks[c]} col_len[c,h] - (col_blocks[c] - k) + 1;

var y {r in ROWS, k in 1..row_blocks[r], ROW_STARTS[r,k]} binary;  # row block k starts at p
var x {c in COLS, k in 1..col_blocks[c], COL_STARTS[c,k]} binary;  # column block k starts at p
var z {ROWS, COLS} binary;  # 1 if the cell is black

subject to OneRowStart {r in ROWS, k in 1..row_blocks[r]}:  # each row block starts once
    sum {p in ROW_STARTS[r,k]} y[r,k,p] = 1;
subject to OneColStart {c in COLS, k in 1..col_blocks[c]}:  # each column block starts once
    sum {p in COL_STARTS[c,k]} x[c,k,p] = 1;
subject to RowOrder {r in ROWS, k in 1..row_blocks[r]-1}:  # next block after this one plus a gap
    sum {p in ROW_STARTS[r,k+1]} p * y[r,k+1,p]
        >= sum {p in ROW_STARTS[r,k]} p * y[r,k,p] + row_len[r,k] + 1;
subject to ColOrder {c in COLS, k in 1..col_blocks[c]-1}:  # next block after this one plus a gap
    sum {p in COL_STARTS[c,k+1]} p * x[c,k+1,p]
        >= sum {p in COL_STARTS[c,k]} p * x[c,k,p] + col_len[c,k] + 1;
subject to RowCover {r in ROWS, c in COLS}:  # black cells match the row placements
    z[r,c] = sum {k in 1..row_blocks[r], p in ROW_STARTS[r,k]:
                  p <= c < p + row_len[r,k]} y[r,k,p];
subject to ColCover {r in ROWS, c in COLS}:  # black cells match the column placements
    z[r,c] = sum {k in 1..col_blocks[c], p in COL_STARTS[c,k]:
                  p <= r < p + col_len[c,k]} x[c,k,p];

param use_cut binary default 0;  # 1 turns on the cell-pattern cut
param First {ROWS, COLS} binary default 0;  # first image found, 1 for black
subject to DifferentImage {if use_cut = 1}:  # no-good cut: differ from First in some cell
    sum {r in ROWS, c in COLS: First[r,c] = 1} z[r,c]
  + sum {r in ROWS, c in COLS: First[r,c] = 0} (1 - z[r,c]) <= R * C - 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
row_clues = [[2, 1], [2], [4], [1, 1]]
col_clues = [[1, 2], [3], [2], [1, 2]]
ampl.param["R"] = len(row_clues)
ampl.param["C"] = len(col_clues)
ampl.param["row_blocks"] = {r + 1: len(b) for r, b in enumerate(row_clues)}
ampl.param["col_blocks"] = {c + 1: len(b) for c, b in enumerate(col_clues)}
ampl.param["row_len"] = {
    (r + 1, k + 1): L for r, b in enumerate(row_clues) for k, L in enumerate(b)
}
ampl.param["col_len"] = {
    (c + 1, k + 1): L for c, b in enumerate(col_clues) for k, L in enumerate(b)
}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
z = {k: round(v) for k, v in ampl.var["z"].to_dict().items()}
for r in range(1, 5):
    print("".join("#" if z[r, c] else "." for c in range(1, 5)))
print("ROW_STARTS[2,1]:", ampl.set["ROW_STARTS"][2, 1].to_list())
ampl.param["First"] = z
ampl.param["use_cut"] = 1
ampl.solve(solver="highs", verbose=False)
print("with the cell-pattern cut:", ampl.solve_result)

solved
##.#
.##.
####
#..#
ROW_STARTS[2,1]: [1, 2, 3]
with the cell-pattern cut: infeasible


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param R := 4;
param C := 4;
param row_blocks := 1 2  2 1  3 1  4 2;
param col_blocks := 1 2  2 1  3 1  4 2;
param row_len := [1,*] 1 2  2 1  [2,*] 1 2  [3,*] 1 4  [4,*] 1 1  2 1;
param col_len := [1,*] 1 1  2 2  [2,*] 1 3  [3,*] 1 2  [4,*] 1 1  2 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
solve;
display solve_result;
for {r in ROWS} {
    printf {c in COLS} "%s", if z[r,c] > 0.5 then "#" else ".";
    printf "\n";
}

solve_result = solved

##.#
.##.
####
#..#


### Another way in amplpy

The cut model is infeasible, so this 4 × 4 picture is unique.

The run below, with the same model.mod and data.py, first prints the start domains of row 1, clue `[2,1]`. It then loads three other clue sets: `ampl.eval("reset data;")` discards the old instance, through `ampl.eval` because amplpy has no method that resets all data. The 2 × 2 puzzle is solved once more with its first image excluded:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
row_clues = [[2, 1], [2], [4], [1, 1]]
col_clues = [[1, 2], [3], [2], [1, 2]]
ampl.param["R"] = len(row_clues)
ampl.param["C"] = len(col_clues)
ampl.param["row_blocks"] = {r + 1: len(b) for r, b in enumerate(row_clues)}
ampl.param["col_blocks"] = {c + 1: len(b) for c, b in enumerate(col_clues)}
ampl.param["row_len"] = {
    (r + 1, k + 1): L for r, b in enumerate(row_clues) for k, L in enumerate(b)
}
ampl.param["col_len"] = {
    (c + 1, k + 1): L for c, b in enumerate(col_clues) for k, L in enumerate(b)
}

In [11]:
# run.py
def load(row_clues, col_clues):
    ampl.eval("reset data;")
    ampl.param["R"] = len(row_clues)
    ampl.param["C"] = len(col_clues)
    ampl.param["row_blocks"] = {r + 1: len(b) for r, b in enumerate(row_clues)}
    ampl.param["col_blocks"] = {c + 1: len(b) for c, b in enumerate(col_clues)}
    ampl.param["row_len"] = {
        (r + 1, k + 1): L for r, b in enumerate(row_clues) for k, L in enumerate(b)
    }
    ampl.param["col_len"] = {
        (c + 1, k + 1): L for c, b in enumerate(col_clues) for k, L in enumerate(b)
    }


def show(label):
    ampl.solve(solver="highs", verbose=False)
    print(f"{label}: {ampl.solve_result}")
    if ampl.solve_result == "solved":
        z = {k: round(v) for k, v in ampl.var["z"].to_dict().items()}
        R, C = int(ampl.param["R"].value()), int(ampl.param["C"].value())
        for r in range(1, R + 1):
            print("  " + "".join("#" if z[r, c] else "." for c in range(1, C + 1)))
        return z


print("ROW_STARTS[1,1]:", ampl.set["ROW_STARTS"][1, 1].to_list())
print("ROW_STARTS[1,2]:", ampl.set["ROW_STARTS"][1, 2].to_list())
load([[2, 1], [], [4], [1, 1]], [[1, 1], [1, 1], [1], [1, 2]])
show("rows 9 black, columns 8 black")
load([[2, 1], [], [4], [1, 1]], [[1, 2], [1, 1], [1], [1, 2]])
show("row 2 empty")
load([[1], [1]], [[1], [1]])
ampl.param["First"] = show("2 x 2, every clue [1]")
ampl.param["use_cut"] = 1
show("2 x 2 with DifferentImage")

ROW_STARTS[1,1]: [1]
ROW_STARTS[1,2]: [4]
rows 9 black, columns 8 black: infeasible
row 2 empty: solved
  ##.#
  ....
  ####
  #..#
2 x 2, every clue [1]: solved
  .#
  #.
2 x 2 with DifferentImage: solved
  #.
  .#


{(1, 1): 1, (1, 2): 0, (2, 1): 0, (2, 2): 1}